# Samian ware: discovery sites and their Pleiades links

This local Jupyter notebook queries the NFDI4Objects Knowledge Graph for Samian-ware **discovery sites** — places where terra sigillata has been found — and checks which of them are linked to [Pleiades](https://pleiades.stoa.org/), the gazetteer of ancient places. The result is visualised as a two-layer `folium` map that makes the coverage of Pleiades identifiers geographically visible.

A companion browser-executable notebook, `n4okg-samian-discovery-sites-live.qmd`, runs the same pipeline via Pyodide and Leaflet directly in the browser for readers who prefer a server-less environment.


## Requirements

```bash
pip install SPARQLWrapper pandas matplotlib folium
```


## About this notebook

Samian ware travelled widely. Finds have been recorded from Britain to North Africa and from the Atlantic coast to the Danube limes. The NFDI4Objects Knowledge Graph represents each find location as a `lado:DiscoverySite` with a GeoSPARQL geometry. Some — but not all — discovery sites additionally carry a `lado:pleiadesID` link into the Pleiades gazetteer, giving a stable external identifier for the ancient place.

### Why this dataset?

The Pleiades-coverage question is a good example of a broader theme in linked open data: *optional* properties are where the heterogeneity lives. A `DiscoverySite` with a Pleiades link participates in the wider ancient-world LOD ecosystem; one without is effectively an island. Making that distinction geographically visible — rather than hiding it behind a single percentage figure — turns a provenance question into a research question about *which* regions are well connected.

### What you'll learn

- how `OPTIONAL` in SPARQL behaves in the result set (missing bindings, not `null` columns)
- how to compute and visualise LOD coverage as a categorical map
- how to attach external-gazetteer links directly into marker popups

### Data-context notes

- one row per discovery site; the `pleiadesID` column is empty when the site has no Pleiades link
- `pleiadesID` is filtered to IRIs (`isIRI`) so that accidental string values are excluded
- coordinates come from `geosparql:hasGeometry` → `geosparql:asWKT` as usual — WKT in `POINT(lon lat)` form
- coordinate precision varies: some sites have exact locations, others are geocoded to a modern settlement or province centroid, which means stacked markers at e.g. a city centre are expected and not a bug

### Tooling notes

Locally we use `SPARQLWrapper` and `folium`, as in the other notebooks of this series. With around 3 900 markers, folium's inline HTML can become large (several megabytes); for even bigger datasets consider `folium.plugins.FastMarkerCluster` or switching to a tile-based vector layer.


## Step 1 — Define the SPARQL query

`OPTIONAL` wraps the Pleiades-link pattern, so a site without a link still appears in the result — just without a `pleiadesID` binding. The `FILTER(isIRI(?pleiadesID))` clause defends against non-IRI values that may slip in during cataloguing.


In [ ]:
from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd

USER_AGENT = "OER-Quarto-Notebook/1.0 (https://n4o-rse.github.io/oer-ta2-koeln/)"
SPARQL_ENDPOINT = "https://graph.nfdi4objects.net/api/sparql"

SPARQL_QUERY = """
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX lado: <http://archaeology.link/ontology#>
SELECT ?item ?itemLabel ?pleiadesID ?geo WHERE {
  GRAPH <https://graph.nfdi4objects.net/collection/8> {
    ?item a <http://archaeology.link/ontology#DiscoverySite> .
    ?item <http://www.opengis.net/ont/geosparql#hasGeometry> ?item_geom .
    ?item rdfs:label ?itemLabel .
    OPTIONAL {
      ?item lado:pleiadesID ?pleiadesID .
      FILTER(isIRI(?pleiadesID))
    }
    ?item_geom <http://www.opengis.net/ont/geosparql#asWKT> ?geo .
  }
}
"""

sparql = SPARQLWrapper(SPARQL_ENDPOINT, agent=USER_AGENT)
sparql.setQuery(SPARQL_QUERY)
sparql.setReturnFormat(JSON)
print("✓ Helpers defined. Run the next cell to load the data.")


## Step 2 — Load the data

A missing `OPTIONAL` variable is simply absent from the bindings dictionary — we use `.get()` to turn that into `None` in the DataFrame. A derived boolean column `has_pleiades` makes subsequent analysis easy.


In [ ]:
import re

_WKT_POINT_RE = re.compile(r"point\s*\(\s*([-+\d.eE]+)\s+([-+\d.eE]+)\s*\)",
                           re.IGNORECASE)

def parse_wkt_point(wkt):
    """Parse 'POINT(lon lat)' (any case) into (lat, lon) ready for mapping libraries."""
    if not wkt:
        return (None, None)
    m = _WKT_POINT_RE.search(wkt)
    if not m:
        return (None, None)
    try:
        lon, lat = float(m.group(1)), float(m.group(2))
        return (lat, lon)
    except ValueError:
        return (None, None)


In [ ]:
bindings = sparql.query().convert()["results"]["bindings"]

rows = []
for b in bindings:
    lat, lon = parse_wkt_point(b.get("geo", {}).get("value"))
    rows.append({
        "item": b["item"]["value"],
        "itemLabel": b["itemLabel"]["value"],
        "pleiadesID": b.get("pleiadesID", {}).get("value"),  # None if absent
        "latitude": lat,
        "longitude": lon,
    })

df = pd.DataFrame(rows)
df = df.dropna(subset=["latitude", "longitude"])
df["has_pleiades"] = df["pleiadesID"].notna()

print(f"Loaded {len(df)} discovery sites — "
      f"{df['has_pleiades'].sum()} with Pleiades link "
      f"({100 * df['has_pleiades'].mean():.1f} %)")
df.head()


## Step 3a — Pleiades coverage (sanity check)

A two-bar summary before the map: how many discovery sites participate in the Pleiades cross-link ecosystem, and how many do not. The raw percentage is useful, but the absolute counts matter too — a high percentage on a small dataset is less reassuring than a middling percentage on a large one.


In [ ]:
import matplotlib.pyplot as plt

counts = df["has_pleiades"].value_counts()
labels = ["With Pleiades link" if k else "Without Pleiades link" for k in counts.index]
colors = ["#2e7d32" if k else "#9e9e9e" for k in counts.index]

fig, ax = plt.subplots(figsize=(7, 3.5))
bars = ax.bar(labels, counts.values, color=colors)
ax.set_ylabel("Number of discovery sites")
ax.set_title(f"Pleiades coverage across {len(df)} Samian-ware discovery sites")
for bar, v in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 20,
            f"{int(v)}\n({100 * v / len(df):.1f} %)",
            ha="center", va="bottom")
ax.set_ylim(0, counts.max() * 1.15)
plt.tight_layout()
plt.show()


## Step 3b — Map: sites with and without Pleiades link

Two map layers, both toggleable via the layer control: sites with a Pleiades link in green, sites without in grey. Popups for Pleiades-linked sites include a direct link out to the Pleiades page, turning the map into a clickable gateway into the ancient-places LOD ecosystem.


In [ ]:
import folium

COLOR_WITH = "#2e7d32"
COLOR_WITHOUT = "#9e9e9e"

lat_c = float(df["latitude"].mean())
lon_c = float(df["longitude"].mean())

m = folium.Map(location=[lat_c, lon_c], zoom_start=4, tiles=None)
folium.TileLayer("OpenStreetMap", name="OpenStreetMap").add_to(m)
folium.TileLayer(
    "https://{s}.tile.openstreetmap.fr/hot/{z}/{x}/{y}.png",
    name="OpenStreetMap.HOT", attr="© OpenStreetMap contributors, HOT style"
).add_to(m)
folium.TileLayer(
    "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    name="Esri.WorldImagery", attr="Tiles © Esri — Source: Esri, Maxar, Earthstar Geographics"
).add_to(m)
folium.TileLayer(
    "https://server.arcgisonline.com/ArcGIS/rest/services/World_Terrain_Base/MapServer/tile/{z}/{y}/{x}",
    name="Esri.WorldTerrain", attr="Tiles © Esri — Source: USGS, Esri, TANA, DeLorme, NPS", max_zoom=13
).add_to(m)

n_with = int(df["has_pleiades"].sum())
n_without = int((~df["has_pleiades"]).sum())
layer_with = folium.FeatureGroup(name=f"With Pleiades link ({n_with})").add_to(m)
layer_without = folium.FeatureGroup(name=f"Without Pleiades link ({n_without})").add_to(m)

for _, row in df.iterrows():
    if row["has_pleiades"]:
        popup_html = (
            f"<b>{row['itemLabel']}</b><br>"
            f'<a href="{row["pleiadesID"]}" target="_blank">Pleiades ↗</a><br>'
            f'<a href="{row["item"]}" target="_blank">N4O KG</a>'
        )
        folium.CircleMarker(
            location=[row["latitude"], row["longitude"]],
            radius=4, color=COLOR_WITH, weight=1,
            fill=True, fill_color=COLOR_WITH, fill_opacity=0.7,
            popup=folium.Popup(popup_html, max_width=300),
        ).add_to(layer_with)
    else:
        popup_html = (
            f"<b>{row['itemLabel']}</b><br>"
            f'<a href="{row["item"]}" target="_blank">N4O KG</a>'
        )
        folium.CircleMarker(
            location=[row["latitude"], row["longitude"]],
            radius=4, color=COLOR_WITHOUT, weight=1,
            fill=True, fill_color=COLOR_WITHOUT, fill_opacity=0.55,
            popup=folium.Popup(popup_html, max_width=300),
        ).add_to(layer_without)

folium.LayerControl(collapsed=True).add_to(m)
m


## Step 4 — Explore

The `df` DataFrame stays in scope — modify the cell below to filter, aggregate, or sanity-check the Pleiades coverage in different ways.


In [ ]:
# Hier anpassen: list unlinked sites in a lat band, inspect Pleiades IRIs, …
df[df["has_pleiades"]].head(10)[["itemLabel", "pleiadesID"]]


---

*Part of an Open Educational Resource series on knowledge graphs and linked open data, produced in the context of [NFDI4Objects](https://www.nfdi4objects.net/).*
